<a href="https://colab.research.google.com/github/R123456-123/ai-eng-safety-alignment/blob/main/ai-engineering/%20notebooks/01_tokenstreaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi uvicorn httpx

In [2]:
%%writefile app.py
import asyncio
import json
import logging
from typing import AsyncGenerator
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ai_gateway")

app = FastAPI(title="The stream of Tokens.")

async def mock_llm_stream_generator(prompt: str, request: Request) -> AsyncGenerator[str, None]:
  """
  Simulates token-by-token generation while checking for client disconnects.
  """
  tokens = f"Simulated response generation for prompt: '{prompt}'. Breaking response into token stream.".split()

  try:

    for i, token in enumerate(tokens):
      #checking that client didn't go off mid-streaming
      if await request.is_disconnected():
        logger.warning(f"Client disconnected early at token index {i}.")
        break

      #simulating latency per generated token
      await asyncio.sleep(0.1)

      #standard SSE format data: <json>
      data_payload = json.dumps({"token_id": i, "text": token + " "})
      yield f"data: {data_payload}\n\n"

    #signal completed
    yield "data: [DONE]\n\n"

  except asyncio.CancelledError:
    logger.info("Stream cancelled by server.")
    raise

@app.post("/v1/chat/stream")
async def stream_chat_completion(request: Request, payload: dict):
  prompt = payload.get("prompt", "")

  return StreamingResponse(
      mock_llm_streaming_generator(prompt, request),
      media_type="text/event-stream",
      headers={
          "Cache-control": "no-cache",
          "Connection": "keep=alive",
          "X-Accel-Buffering": "no", #prevent proxy buffering
      }
  )

Writing app.py


In [3]:
import subprocess
import time

# Launch Uvicorn server on port 8000 in background
server_process = subprocess.Popen(["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"])

# Give server 2 seconds to initialize
time.sleep(2)
print("FastAPI Gateway running in background on http://127.0.0.1:8000")

FastAPI Gateway running in background on http://127.0.0.1:8000


In [4]:
import httpx
import json

async def test_stream():
    url = "http://127.0.0.1:8000/v1/chat/stream"
    payload = {"prompt": "Explain LLM quantization in safety"}

    print("--- Stream Start ---\n")
    async with httpx.AsyncClient() as client:
        async with client.stream("POST", url, json=payload) as response:
            async for line in response.aiter_lines():
                if line.startswith("data: "):
                    raw_data = line.replace("data: ", "").strip()
                    if raw_data == "[DONE]":
                        print("\n\n--- Stream Complete ---")
                        break

                    data = json.loads(raw_data)
                    # Print tokens in real-time without newline buffering
                    print(data["text"], end="", flush=True)

# Run the async consumer in Colab
await test_stream()

--- Stream Start ---



In [5]:
server_process.terminate()
print("Server terminated.")

Server terminated.
